# IT Fusion — Dataset Preparation
## Supports: DERM1M, MM-Skin, PAD-UFES-20
Converts raw CSVs to unified parquet format for MARIA fusion training.

In [1]:
import os, sys, json
import pandas as pd
import numpy as np
import torch
from PIL import Image

from tqdm import tqdm
from pathlib import Path
from torchvision import transforms
from transformers import AutoTokenizer, AutoModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cuda


## Config — set your paths here

In [2]:
# ── Paths (edit these to match your system) ──
BASE = Path('/data/Stagewise Dataset/IT Fusion')

DERM1M_CSV   = BASE / 'Derm1M/Derm1M_v2_pretrain.csv'
DERM1M_IMG   = BASE / 'Derm1M'

# MM-SkinQA — folder is 'MM-SkinQA', CSV is 'caption.csv', images under 'images/dataset/'
MMSKIN_CSV   = BASE / 'MM-SkinQA/caption.csv'
MMSKIN_IMG   = BASE / 'MM-SkinQA/images'

# PAD-UFES-20 — folder is 'PAD-UFES-20', CSV is 'metadata.csv', images under 'images/'
PADUFES_CSV  = BASE / 'PAD-UFES-20/metadata.csv'
PADUFES_IMG  = BASE / 'PAD-UFES-20/images'

OUT_DIR = BASE / 'prepared'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUT_DIR)
print('DERM1M CSV exists:', DERM1M_CSV.exists())
print('MM-SkinQA CSV exists:', MMSKIN_CSV.exists())
print('PAD-UFES CSV exists:', PADUFES_CSV.exists())

Output dir: /data/Stagewise Dataset/IT Fusion/prepared
DERM1M CSV exists: True
MM-SkinQA CSV exists: True
PAD-UFES CSV exists: True


## Shared Encoders
- **Text:** Bio_ClinicalBERT (fine-tuned in Stage 5 — MedProc) → 768-dim [CLS] embeddings
- **Image:** EfficientFormerV2-S2 backbone (fine-tuned in Stage 4 — HC) → 288-dim features
  Loaded via `hc_model.py` → `load_hc_backbone()`

In [3]:
# ════════════════════════════════════════════════════════════════════
# TEXT ENCODER — Bio_ClinicalBERT (Stage 5: MedProc)
# ════════════════════════════════════════════════════════════════════
tokenizer = AutoTokenizer.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')
text_model = AutoModel.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')

medproc_ckpt = '/data/Stagewise Dataset/MedProc/checkpoints/medproc_best.pt'
if os.path.exists(medproc_ckpt):
    ckpt = torch.load(medproc_ckpt, map_location=DEVICE, weights_only=False)  # tokenizer object requires this (PyTorch 2.6+)
    sd = ckpt.get('model_state', ckpt)
    bert_sd = {k.replace('bert.', ''): v for k, v in sd.items() if k.startswith('bert.')}
    text_model.load_state_dict(bert_sd if bert_sd else sd, strict=False)
    print('✓ Loaded fine-tuned Bio_ClinicalBERT weights from MedProc (Stage 5)')
else:
    print('⚠ MedProc checkpoint not found — using base Bio_ClinicalBERT weights')
text_model = text_model.to(DEVICE).eval()

# ════════════════════════════════════════════════════════════════════
# IMAGE ENCODER — EfficientFormerV2-S2 backbone (Stage 4: HC)
# Imported from hc_model.py — single source of truth for HC architecture
# ════════════════════════════════════════════════════════════════════
FUSION_DIR = Path('/home/vjti-comp/Desktop/Final Project Code/it_fusion_modular_project')
sys.path.insert(0, str(FUSION_DIR))
from hc_model import load_hc_backbone

img_model = load_hc_backbone(device=DEVICE)

img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print(f'\nText encoder  : Bio_ClinicalBERT → 768-dim')
print(f'Image encoder : EfficientFormerV2-S2 backbone → 288-dim')
print('Models loaded')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Loaded fine-tuned Bio_ClinicalBERT weights from MedProc (Stage 5)
✓ Loaded HC backbone from hc_best_v2.pt

Text encoder  : Bio_ClinicalBERT → 768-dim
Image encoder : EfficientFormerV2-S2 backbone → 288-dim
Models loaded


In [4]:
# ════════════════════════════════════════════════════════════════════
# ABCDE COMPUTATION — ARCUNet (Stage 2) + SLRC (Stage 3)
# Runs per image to produce real ABCDE values at dataset prep time
# ════════════════════════════════════════════════════════════════════
import cv2
sys.path.insert(0, str(FUSION_DIR))
from abcde_computation import compute_abcde, ABCDEResult
from abcde_inference   import abcde_to_feature_vector

# ── Load ARCUNet segmentation model (Stage 2) ─────────────────────────
ARCUNET_CKPT = Path('/home/vjti-comp/Desktop/Final Project Code/SLSf/arcunet_best_v3.pt')

arcunet_model = None
try:
    from arcunet import ARCUNet   # import from Stage 2 module
    arcunet_model = ARCUNet().to(DEVICE)
    if ARCUNET_CKPT.exists():
        arcunet_model.load_state_dict(
            torch.load(ARCUNET_CKPT, map_location=DEVICE, weights_only=True)
        )
        print('✓ ARCUNet loaded from checkpoint')
    else:
        print('⚠ ARCUNet checkpoint not found — ABCDE will use fallback mask')
    arcunet_model.eval()
except Exception as e:
    print(f'⚠ ARCUNet not available: {e}')
    arcunet_model = None

# ── Segmentation transform (512×512, normalised) ──────────────────────
seg_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


@torch.no_grad()
def get_mask(img_path) -> np.ndarray:
    """
    Run ARCUNet on the image to produce a binary segmentation mask.
    Returns: (512, 512) uint8 mask — 255 = lesion, 0 = background.
    Falls back to a full-image mask if ARCUNet is unavailable.
    """
    try:
        img = Image.open(img_path).convert('RGB')
        t   = seg_transform(img).unsqueeze(0).to(DEVICE)
        if arcunet_model is not None:
            pred = arcunet_model(t)           # (1, 1, 512, 512)
            mask = (pred.squeeze().cpu().numpy() > 0.5).astype(np.uint8) * 255
        else:
            # Fallback: assume entire 512×512 image is the lesion
            mask = np.ones((512, 512), dtype=np.uint8) * 255
        return mask
    except Exception:
        return np.ones((512, 512), dtype=np.uint8) * 255


def get_roi(img_path, mask: np.ndarray) -> np.ndarray:
    """
    SLRC: crop the bounding box of the lesion from the mask,
    then resize to 224×224 RGB for downstream ABCDE colour analysis.
    Returns: (224, 224, 3) uint8 RGB array.
    """
    try:
        img  = np.array(Image.open(img_path).convert('RGB'))  # H×W×3 RGB
        m    = (mask > 127).astype(np.uint8)
        ys, xs = np.where(m > 0)
        if len(xs) == 0:
            roi = img
        else:
            y1, y2 = ys.min(), ys.max() + 1
            x1, x2 = xs.min(), xs.max() + 1
            # Scale bbox coords to original image size
            h_orig, w_orig = img.shape[:2]
            y1 = int(y1 * h_orig / mask.shape[0])
            y2 = int(y2 * h_orig / mask.shape[0])
            x1 = int(x1 * w_orig / mask.shape[1])
            x2 = int(x2 * w_orig / mask.shape[1])
            roi = img[y1:y2, x1:x2]
        roi_224 = cv2.resize(roi, (224, 224), interpolation=cv2.INTER_LINEAR)
        return roi_224   # RGB uint8 (224×224×3)
    except Exception:
        return np.zeros((224, 224, 3), dtype=np.uint8)


def compute_abcde_for_image(
    img_path,
    evolution_score: float = 0.0,
    symptom_keywords: list = None,
    pixel_spacing_mm: float = 0.1,
) -> dict:
    """
    Full ABCDE computation for one image.

    1. ARCUNet  → binary segmentation mask (512×512)
    2. SLRC     → bounding-box ROI crop   (224×224 RGB)
    3. compute_abcde() → ABCDEResult
    4. abcde_to_feature_vector() → 13-dim feature dict

    Returns
    -------
    {
      'abcde_vec'  : list[float]  — 13 features, non-zero when image is valid
      'abcde_mask' : list[float]  — 13 ones if computed, 13 zeros on failure
    }
    """
    try:
        mask    = get_mask(img_path)                    # (512, 512) uint8
        roi_rgb = get_roi(img_path, mask)               # (224, 224, 3) RGB

        # Also resize mask to match ROI for compute_abcde
        mask_224 = cv2.resize(mask, (224, 224), interpolation=cv2.INTER_NEAREST)

        result = compute_abcde(
            mask=mask_224,
            roi_img=roi_rgb,            # RGB — abcde_computation.py now handles RGB
            evolution_score=evolution_score,
            symptom_keywords=symptom_keywords or [],
            pixel_spacing_mm=pixel_spacing_mm,
        )
        feats = abcde_to_feature_vector(result)         # 13-key dict
        return {
            'abcde_vec':  list(feats.values()),          # [A,B,C,D,E,risk,A_g,...]
            'abcde_mask': [1.0] * 13,                   # fully present
        }
    except Exception as e:
        return {
            'abcde_vec':  [0.0] * 13,
            'abcde_mask': [0.0] * 13,   # masked out — tells MARIA to ignore
        }


print('✓ ABCDE helpers defined')
print('  get_mask()              — ARCUNet segmentation → (512×512) binary mask')
print('  get_roi()               — SLRC bbox crop       → (224×224) RGB array')
print('  compute_abcde_for_image() — full pipeline → 13-dim ABCDE feature vector')


⚠ ARCUNet not available: No module named 'arcunet'
✓ ABCDE helpers defined
  get_mask()              — ARCUNet segmentation → (512×512) binary mask
  get_roi()               — SLRC bbox crop       → (224×224) RGB array
  compute_abcde_for_image() — full pipeline → 13-dim ABCDE feature vector


In [5]:
@torch.no_grad()
def encode_image(path):
    """Extract 288-dim feature vector using HC's EfficientFormerV2-S2 backbone."""
    try:
        img = Image.open(path).convert('RGB')
        t = img_transform(img).unsqueeze(0).to(DEVICE)
        feats = img_model(t)
        if isinstance(feats, (list, tuple)):
            feats = feats[0]
        return feats.squeeze(0).cpu().numpy().tolist()
    except:
        return [0.0] * 288

@torch.no_grad()
def encode_text(text):
    """Extract 768-dim [CLS] embedding using MedProc's Bio_ClinicalBERT."""
    try:
        enc = tokenizer(str(text), truncation=True, padding=True,
                        max_length=512, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        out = text_model(**enc)
        return out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy().tolist()
    except:
        return [0.0] * 768

def safe_float(v, default=-1.0):
    try: return float(v)
    except: return default

def encode_gender(g):
    s = str(g).lower()
    return 1 if 'male' in s and 'female' not in s else (2 if 'female' in s else 0)

print('Helpers defined')

Helpers defined


## 1. DERM1M
Columns: `filename, caption, truncated_caption, source, source_type,`
`disease_label, hierarchical_disease_label, skin_concept, body_location, symptoms, age, gender`

In [6]:
derm1m = pd.read_csv(DERM1M_CSV)
print('DERM1M shape:', derm1m.shape)
print(derm1m.head(2).to_string())

DERM1M shape: (413210, 12)
                                filename                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 caption                                                                                                                                                                                                                                                                                                                                                                                                                                                                       trunca

In [ ]:
# Build label map from disease_label column
all_labels = sorted(derm1m['disease_label'].dropna().unique())
d1m_label2id = {l:i for i,l in enumerate(all_labels)}
print(f'DERM1M classes: {len(d1m_label2id)}')

records = []
for _, row in tqdm(derm1m.iterrows(), total=len(derm1m), desc='DERM1M'):
    img_path  = DERM1M_IMG / row['filename']
    text      = str(row.get('truncated_caption', row.get('caption', '')))
    label_str = str(row.get('disease_label', 'unknown'))
    abcde     = compute_abcde_for_image(img_path)  # ARCUNet → SLRC → ABCDE
    records.append({
        'dataset':        'DERM1M',
        'image_path':     str(img_path),
        'text':           text,
        'disease_label':  label_str,
        'target_class':   d1m_label2id.get(label_str, -1),
        'age':            safe_float(row.get('age', -1)),
        'gender':         encode_gender(row.get('gender', '')),
        'modality':       str(row.get('source_type', 'forum')),
        'body_location':  str(row.get('body_location', '')),
        'image_features': encode_image(img_path),
        'text_features':  encode_text(text),
        'abcde_vec':      abcde['abcde_vec'],   # 13-dim — from ARCUNet+SLRC
        'abcde_mask':     abcde['abcde_mask'],  # 13 ones if valid, 13 zeros on failure
    })

derm1m_df = pd.DataFrame(records)
out1 = OUT_DIR / 'derm1m_prepared.parquet'
derm1m_df.to_parquet(out1, index=False)
print(f'Saved {len(derm1m_df)} rows → {out1}')
print(f'  ABCDE computed: {(derm1m_df["abcde_mask"].apply(lambda x: x[0]) == 1.0).sum()} / {len(derm1m_df)}')


DERM1M classes: 22708


DERM1M:   1%|▏         | 5775/413210 [08:05<8:04:07, 14.03it/s] 

## 2. MM-Skin
Columns: `image, caption, modality, sex, age, cleaned_caption`

In [ ]:
mmskin = pd.read_csv(MMSKIN_CSV)
print('MM-Skin shape:', mmskin.shape)
print(mmskin.head(2).to_string())

MM-Skin shape: (11090, 6)
                 image                                                                                                                                                                                                                                                                                                                                    caption modality  sex  age                                                                                                                                                                                                                                                                                                                            cleaned_caption
0  dataset/bk2_2_1.png  Cutaneousn/8 T-cell lymphoma. Multiple erythematous plaques and tumors on the trunk and face. This patient was included within the“subcutaneous T-cell lymphomas” in the first edition of this book(Figs 7.1 and 7.2 of that edition), but regasificat

In [ ]:
# MM-SkinQA — primarily text/captioning, no discrete disease label
records2 = []
for _, row in tqdm(mmskin.iterrows(), total=len(mmskin), desc='MM-Skin'):
    img_path = MMSKIN_IMG / str(row['image'])
    text     = str(row.get('cleaned_caption', row.get('caption', '')))
    abcde    = compute_abcde_for_image(img_path)
    records2.append({
        'dataset':        'MM-Skin',
        'image_path':     str(img_path),
        'text':           text,
        'disease_label':  'unknown',
        'target_class':   -1,
        'age':            safe_float(row.get('age', -1)),
        'gender':         encode_gender(row.get('sex', '')),
        'modality':       str(row.get('modality', 'skin')),
        'body_location':  '',
        'image_features': encode_image(img_path),
        'text_features':  encode_text(text),
        'abcde_vec':      abcde['abcde_vec'],
        'abcde_mask':     abcde['abcde_mask'],
    })

mmskin_df = pd.DataFrame(records2)
out2 = OUT_DIR / 'mmskin_prepared.parquet'
mmskin_df.to_parquet(out2, index=False)
print(f'Saved {len(mmskin_df)} rows → {out2}')
print(f'  ABCDE computed: {(mmskin_df["abcde_mask"].apply(lambda x: x[0]) == 1.0).sum()} / {len(mmskin_df)}')


MM-Skin: 100%|██████████| 11090/11090 [00:39<00:00, 279.42it/s]


Saved 11090 rows → /data/Stagewise Dataset/IT Fusion/prepared/mmskin_prepared.parquet


## 3. PAD-UFES-20
Columns include: `patient_id, lesion_id, age, gender, fitspatrick, region,`
`diameter_1, diameter_2, diagnostic, itch, grew, hurt, changed, bleed, elevation, img_id, biopsed`

Labels: NEV, MEL, BCC, SCC, ACK, SEK (6 classes, all confirmed by biopsy)

In [ ]:
padufes = pd.read_csv(PADUFES_CSV)
print('PAD-UFES-20 shape:', padufes.shape)
print(padufes.head(2).to_string())

PAD-UFES-20 shape: (2298, 26)
  patient_id  lesion_id  smoke  drink background_father background_mother  age pesticide  gender skin_cancer_history cancer_history has_piped_water has_sewage_system  fitspatrick region  diameter_1  diameter_2 diagnostic   itch   grew   hurt changed  bleed elevation                 img_id  biopsed
0   PAT_1516       1765    NaN    NaN               NaN               NaN    8       NaN     NaN                 NaN            NaN             NaN               NaN          NaN    ARM         NaN         NaN        NEV  False  False  False   False  False     False  PAT_1516_1765_530.png    False
1     PAT_46        881  False  False         POMERANIA         POMERANIA   55     False  FEMALE                True           True            True              True          3.0   NECK         6.0         5.0        BCC   True   True  False    True   True      True     PAT_46_881_939.png     True


In [ ]:
# PAD-UFES has structured tabular data — we build a natural language caption from columns
# and use the diagnostic column as the label

PAD_LABEL_MAP = {'NEV':0,'MEL':1,'BCC':2,'SCC':3,'ACK':4,'SEK':5}

SYMPTOM_COLS = ['itch','grew','hurt','changed','bleed','elevation']

def pad_caption(row):
    """Build a free-text clinical description from PAD-UFES columns."""
    parts = []
    age = row.get('age', '')
    gender = row.get('gender', '')
    if pd.notna(age): parts.append(f'{age}-year-old {gender} patient')
    region = row.get('region', '')
    if pd.notna(region) and str(region).strip(): parts.append(f'lesion on {region}')
    d1, d2 = row.get('diameter_1', ''), row.get('diameter_2', '')
    if pd.notna(d1) and pd.notna(d2):
        parts.append(f'diameter {d1}x{d2}mm')
    symptoms = [c for c in SYMPTOM_COLS if str(row.get(c, '')).lower() == 'true']
    # Fix: avoid nested quotes inside f-string — use a variable
    if symptoms:
        sym_str = ', '.join(symptoms)
        parts.append(f'symptoms: {sym_str}')
    diag = row.get('diagnostic', '')
    if pd.notna(diag): parts.append(f'diagnosis {diag}')
    return '. '.join(parts) if parts else 'Skin lesion'

# Build ABCDE-relevant tabular modality
def pad_tabular(row):
    """Returns 6-dim symptom vector matching fusion.py 'symptoms' modality."""
    return np.array(
        [1.0 if str(row.get(c, '')).lower() == 'true' else 0.0 for c in SYMPTOM_COLS],
        dtype=np.float32
    ).tolist()

def pad_demographics(row):
    """Returns [age_norm, gender_enc, fitzpatrick] — 3-dim demographics vector."""
    age = safe_float(row.get('age', -1))
    age_norm = age / 100.0 if age > 0 else -1.0
    gender = encode_gender(row.get('gender', ''))
    fitz = safe_float(row.get('fitspatrick', -1))
    return [age_norm, float(gender), fitz]

records3 = []
for _, row in tqdm(padufes.iterrows(), total=len(padufes), desc='PAD-UFES-20'):
    img_path = PADUFES_IMG / str(row['img_id'])
    diag = str(row.get('diagnostic', 'unknown')).upper().strip()
    caption = pad_caption(row)
    # PAD-UFES has real clinical Evolution signals — use them for E score
    symptoms    = [c for c in SYMPTOM_COLS if str(row.get(c,'')).lower()=='true']
    evo_kws     = [s for s in symptoms if s in {'grew','bleed','changed'}]
    evo_score   = min(1.0, len(evo_kws) * 0.35)   # simple heuristic from PAD-UFES columns
    abcde       = compute_abcde_for_image(img_path,
                      evolution_score=evo_score, symptom_keywords=symptoms)
    records3.append({
        'dataset': 'PAD-UFES-20',
        'image_path': str(img_path),
        'text': caption,
        'disease_label': diag,
        'target_class': PAD_LABEL_MAP.get(diag, -1),
        'age': safe_float(row.get('age', -1)),
        'gender': encode_gender(row.get('gender', '')),
        'modality': 'clinical',
        'body_location': str(row.get('region', '')),
        'patient_id': str(row.get('patient_id', '')),
        'lesion_id': str(row.get('lesion_id', '')),
        'diameter_1': safe_float(row.get('diameter_1', -1)),
        'diameter_2': safe_float(row.get('diameter_2', -1)),
        'biopsed': str(row.get('biopsed', 'False')).lower() == 'true',
        'symptoms_vec': pad_tabular(row),
        'demographics_vec': pad_demographics(row),
        'image_features': encode_image(img_path),
        'text_features':  encode_text(caption),
        'abcde_vec':  abcde['abcde_vec'],
        'abcde_mask': abcde['abcde_mask'],
    })

padufes_df = pd.DataFrame(records3)
out3 = OUT_DIR / 'padufes_prepared.parquet'
padufes_df.to_parquet(out3, index=False)
print(f'Saved {len(padufes_df)} rows → {out3}')

PAD-UFES-20: 100%|██████████| 2298/2298 [02:00<00:00, 19.03it/s]


Saved 2298 rows → /data/Stagewise Dataset/IT Fusion/prepared/padufes_prepared.parquet


## 4. Merged Dataset (for MARIA training)

In [ ]:
# Load all three prepared parquets and merge on common columns
COMMON = ['dataset','image_path','text','disease_label','target_class',
          'age','gender','modality','body_location',
          'image_features','text_features',
          'abcde_vec','abcde_mask']  # ABCDE computed per-image by ARCUNet+SLRC

d1 = pd.read_parquet(OUT_DIR/'derm1m_prepared.parquet')[COMMON]
d2 = pd.read_parquet(OUT_DIR/'mmskin_prepared.parquet')[COMMON]
d3 = pd.read_parquet(OUT_DIR/'padufes_prepared.parquet')[COMMON]

merged = pd.concat([d1, d2, d3], ignore_index=True)
print('Merged shape:', merged.shape)
print(merged['dataset'].value_counts())

Merged shape: (426598, 11)
dataset
DERM1M         413210
MM-Skin         11090
PAD-UFES-20      2298
Name: count, dtype: int64


In [ ]:
# Build MARIA 5-modality feature vectors for each row
import ast

MODALITY_SIZES = {'demographics':3,'symptoms':6,'image_meta':5,'medical_history':8,'abcde':13}

def build_maria_modalities(row, row_extra=None):
    # row_extra: full row dict for any dataset (contains abcde_vec, abcde_mask)
    """Build the 5 modality dicts + masks expected by fusion.py."""
    age_norm = row['age']/100.0 if row['age'] > 0 else -1.0
    mod_enc = {'forum':0,'book':1,'clinical':2,'skin':1}.get(str(row['modality']).lower(),0)

    demographics = np.array([age_norm, float(row['gender']), float(mod_enc)], dtype=np.float32)
    demo_mask    = np.array([1,1,1], dtype=np.float32)

    # Symptoms — use PAD-UFES vector if available, else zeros
    if row_extra and 'symptoms_vec' in row_extra:
        symptoms = np.array(row_extra['symptoms_vec'], dtype=np.float32)
        sym_mask = np.ones(6, dtype=np.float32)
    else:
        symptoms = np.zeros(6, dtype=np.float32)
        sym_mask = np.zeros(6, dtype=np.float32)

    image_meta = np.zeros(5, dtype=np.float32)  # filled by HC at inference
    img_mask   = np.zeros(5, dtype=np.float32)

    med_history = np.zeros(8, dtype=np.float32)  # filled from patient DB at inference
    med_mask    = np.zeros(8, dtype=np.float32)

    # ABCDE was computed per-image by ARCUNet+SLRC during dataset prep
    raw_abcde = row_extra.get('abcde_vec', None) if row_extra else None
    if raw_abcde is not None and isinstance(raw_abcde, (list, np.ndarray)):
        abcde_vec  = np.array(raw_abcde, dtype=np.float32)
        abcde_mask = np.ones(13, dtype=np.float32)
    else:
        abcde_vec  = np.zeros(13, dtype=np.float32)
        abcde_mask = np.zeros(13, dtype=np.float32)

    return {
        'modality_data': {
            'demographics':    demographics.tolist(),
            'symptoms':        symptoms.tolist(),
            'image_meta':      image_meta.tolist(),
            'medical_history': med_history.tolist(),
            'abcde':           abcde_vec.tolist(),
        },
        'feature_masks': {
            'demographics':    demo_mask.tolist(),
            'symptoms':        sym_mask.tolist(),
            'image_meta':      img_mask.tolist(),
            'medical_history': med_mask.tolist(),
            'abcde':           abcde_mask.tolist(),
        }
    }

# Apply to merged
maria_records = []
pad_lookup = padufes_df.set_index('image_path').to_dict('index') if 'padufes_df' in dir() else {}

for _, row in tqdm(merged.iterrows(), total=len(merged), desc='Building MARIA vectors'):
    # row already has abcde_vec/abcde_mask from dataset prep
    # pad_lookup adds symptoms_vec for PAD-UFES rows
    extra = {**dict(row), **(pad_lookup.get(row['image_path'], {}))}
    mods  = build_maria_modalities(row, extra)
    maria_records.append({
        'dataset':        row['dataset'],
        'image_path':     row['image_path'],
        'disease_label':  row['disease_label'],
        'target_class':   int(row['target_class']),
        'image_features': row['image_features'],
        'text_features':  row['text_features'],
        'modality_data':  json.dumps(mods['modality_data']),
        'feature_masks':  json.dumps(mods['feature_masks']),
    })

maria_df = pd.DataFrame(maria_records)
out_maria = OUT_DIR / 'it_fusion_maria_dataset.parquet'
maria_df.to_parquet(out_maria, index=False)
print(f'MARIA dataset saved: {out_maria}')
print(f'Shape: {maria_df.shape}')

Building MARIA vectors: 100%|██████████| 426598/426598 [00:18<00:00, 22812.87it/s]


MARIA dataset saved: /data/Stagewise Dataset/IT Fusion/prepared/it_fusion_maria_dataset.parquet
Shape: (426598, 8)


## 5. PyTorch Dataset Class

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class ITFusionDataset(Dataset):
    def __init__(self, parquet_path, filter_unlabeled=True):
        self.df = pd.read_parquet(parquet_path)
        if filter_unlabeled:
            self.df = self.df[self.df['target_class'] >= 0].reset_index(drop=True)
        print(f'Dataset: {len(self.df)} labeled samples')

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mdata = json.loads(row['modality_data'])
        fmask = json.loads(row['feature_masks'])
        return {
            'image_features': torch.tensor(np.array(row['image_features']), dtype=torch.float32),
            'text_features':  torch.tensor(np.array(row['text_features']),  dtype=torch.float32),
            'modality_data':  {k: torch.tensor(v, dtype=torch.float32) for k,v in mdata.items()},
            'feature_masks':  {k: torch.tensor(v, dtype=torch.float32) for k,v in fmask.items()},
            'target_class':   torch.tensor(int(row['target_class']), dtype=torch.long),
            'disease_label':  row['disease_label'],
            'dataset':        row['dataset'],
        }

dataset = ITFusionDataset(OUT_DIR / 'it_fusion_maria_dataset.parquet')
sample  = dataset[0]
print('Sample keys:', list(sample.keys()))
print('image_features:', sample['image_features'].shape)
print('text_features:', sample['text_features'].shape)

Dataset: 415508 labeled samples
Sample keys: ['image_features', 'text_features', 'modality_data', 'feature_masks', 'target_class', 'disease_label', 'dataset']
image_features: torch.Size([288])
text_features: torch.Size([768])


## 6. Summary Stats

In [ ]:
print('=== Dataset Summary ===')
print(f'Total samples: {len(dataset)}')
print(f'\nBy dataset:')
print(maria_df['dataset'].value_counts())
print(f'\nBy disease label (top 20):')
print(maria_df['disease_label'].value_counts().head(20))
print(f'\nPAD-UFES label distribution:')
print(padufes_df['disease_label'].value_counts())
print(f'\nFiles saved:')
for f in OUT_DIR.glob('*.parquet'):
    print(f'  {f.name}: {f.stat().st_size/1e6:.1f} MB')

=== Dataset Summary ===
Total samples: 415508

By dataset:
dataset
DERM1M         413210
MM-Skin         11090
PAD-UFES-20      2298
Name: count, dtype: int64

By disease label (top 20):
disease_label
no definitive diagnosis                           102380
allergic contact dermatitis                        47510
irritated seborrheic keratosis (from "sk/isk")     15906
psoriasis                                          13794
unknown                                            11090
nevus                                               8706
melanoma                                            8025
eczema                                              6808
keratosis                                           4688
basal cell carcinoma                                4683
dermatitis                                          3888
proliferations                                      3448
squamous cell carcinoma                             3132
urticaria                                           2859
a